# Domain Prediction

Multiclass

In [10]:
import pandas as pd
import numpy as np
import re

# from sklearn.model_selection import train_test_split
# from sklearn import metrics
# from sklearn.metrics import accuracy_score, precision_recall_fscore_support


# Handling html data
# from bs4 import BeautifulSoup

# import seaborn as sns
# from pylab import rcParams
import matplotlib.pyplot as plt
from matplotlib import rc
%matplotlib inline
import json
import ast


## Load data, prepare features

In [11]:
import os

# full text or decision?

# root = 'drive/MyDrive/Legal NLP Project (with MPI Coll)/Colab_coding_files/latest_code'
root = '.'
# df_cleaned_final = pd.read_csv('data/df_cleaned_final_new.csv', index_col=[0])
# df_cleaned = pd.read_csv(os.path.join(root, 'data', 'court_case_data.csv')).drop_duplicates(subset=['uid']) # already filtered to 1 domain?
df_cleaned = pd.read_csv(os.path.join(root, 'data', 'df_cleaned_final_new.csv')).drop_duplicates(subset=['uid']) # already filtered to 1 domain?
print(df_cleaned.shape)
bverfg = pd.read_csv(os.path.join(root, 'venv', 'senate_1', 'bverfg_senate_1.csv'))
# bverfg = pd.read_csv(os.path.join(root, 'data', 'bverfg_chamber_1.csv'))

# df_cleaned_final


(3192, 166)


/var/folders/b_/djy58vhd1s3d4l09hngvbng40000gp/T/ipykernel_23097/4270147309.py:11: DtypeWarning: Columns (84,85,91,1425) have mixed types. Specify dtype option on import or set low_memory=False.
  bverfg = pd.read_csv(os.path.join(root, 'venv', 'senate_1', 'bverfg_senate_1.csv'))


In [13]:
# list(df_cleaned.columns)

['Unnamed: 0',
 'uid',
 'date_and_first_docket',
 'date',
 'year',
 'ref_seidl',
 'ref_papier',
 'ref_haas',
 'ref_grimm',
 'ref_hoemig',
 'ref_kuehling',
 'ref_seibert',
 'ref_jaeger',
 'ref_steiner',
 'ref_hohmanndennhardt',
 'ref_hoffmannriem',
 'ref_bryde',
 'ref_gaier',
 'ref_schluckebier',
 'ref_eichberger',
 'ref_fkirchhof',
 'ref_masing',
 'ref_paulus',
 'ref_britz',
 'ref_baer',
 'ref_ott',
 'ref_christ',
 'ref_radtke',
 'ref_harbarth',
 'ref_haertel',
 'ref_wolff',
 'sumref',
 'ref2_kruis',
 'ref2_grasshof',
 'ref2_pkirchhof',
 'ref2_winter',
 'ref2_limbach',
 'ref2_sommer',
 'ref2_jentsch',
 'ref2_hassemer',
 'ref2_bross',
 'ref2_osterloh',
 'ref2_mellinghoff',
 'ref2_difabio',
 'ref2_luebbewolff',
 'ref2_gerhardt',
 'ref2_landau',
 'ref2_vosskuhle',
 'ref2_hermanns',
 'ref2_huber',
 'ref2_pmueller',
 'ref2_kessalwulf',
 'ref2_koenig',
 'ref2_maidowski',
 'ref2_langenfeld',
 'ref2_wallrabenstein',
 'sumref2',
 'dm_environmental_prob',
 'dm_tax_prob',
 'dm_property_prob',
 'd

In [14]:
# we expect many GPT predictions
# get cases where there are multiple candidate judges, given one GPT domain

bverfg['name'].notnull().sum()
# gpt_predictions = bverfg[bverfg['name'].notnull()]
# gpt_predictions

302

In [16]:

# merge datasets. TODO: filter bverfg for space

df_cleaned_final = bverfg.merge(df_cleaned[[
    'uid', 'full_text_cleaned', 'domains_of_judges', 'ref_name_final',
    'prob','combined_domains_score'
    ]], left_on='id', right_on='uid')
# df_cleaned_final = bverfg[['id','name']].merge(df_cleaned, left_on='id', right_on='uid')
df_cleaned_final

df = df_cleaned_final
df

,Unnamed: 0,id,date_and_first_docket,az,senate_and_chamber,judges,short_description,decision,full_text,url,...,nrow,dm,name,sumref_new,uid,full_text_cleaned,domains_of_judges,ref_name_final,prob,combined_domains_score
0,1,1,30. Dezember 1997 - 1 BvR 2264/97,'1 BvR 2264/97',1. Senat,"['seidl', 'haas', 'hoemig']",Bundesverfassungsgericht - Entscheidungen - ...,NaN,Bundesverfassungsgericht \n- 1 BvR 2264/97 -\n...,https://www.bundesverfassungsgericht.de/Shared...,...,NaN,NaN,NaN,0,1,bundesverfassungsgericht 1 bvr 390/96 in dem v...,"{'jaeger': ['dm_professions', 'dm_exams', 'dm_...",kuehling,0.37,0.125
1,2,2,30. Dezember 1997 - 1 BvR 2339/95,"'1 BvR 2339/95 1 BvR 5/97', ''",1. Senat 1. Kammer,"['seidl', 'haas', 'hoemig']",Bundesverfassungsgericht - Entscheidungen - ...,NaN,Bundesverfassungsgericht \n- 1 BvR 2339/95 - \...,https://www.bundesverfassungsgericht.de/Shared...,...,NaN,NaN,NaN,0,2,bundesverfassungsgericht 1 bvr 1872/94 in dem ...,"{'jaeger': ['dm_professions', 'dm_exams', 'dm_...",kuehling,0.27,0.130
2,25,25,29. Januar 1998 - 1 BvR 2422/97,'1 BvR 2422/97',1. Senat,"['seidl', 'grimm', 'hoemig']",Bundesverfassungsgericht - Entscheidungen -,NaN,Bundesverfassungsgericht \n- 1 BvR 2422/97 -\n...,https://www.bundesverfassungsgericht.de/Shared...,...,NaN,NaN,NaN,2,25,bundesverfassungsgericht 1 bvr 410/95 im namen...,"{'grimm': ['dm_restitution', 'dm_association',...",grimm,0.27,0.200
3,26,26,30. Januar 1998 - 1 BvR 1713/96,'1 BvR 1713/96',1. Senat,"['seidl', 'grimm', 'hoemig']",Bundesverfassungsgericht - Entscheidungen - ...,NaN,Bundesverfassungsgericht \n- 1 BvR 1713/96 -\n...,https://www.bundesverfassungsgericht.de/Shared...,...,NaN,NaN,NaN,1,26,bundesverfassungsgericht 1 bvr 2666/95 im name...,"{'grimm': ['dm_restitution', 'dm_association',...",grimm,0.33,0.220
4,29,29,05. Februar 1998 - 1 BvR 410/95,'1 BvR 410/95',1. Senat,"['grimm', 'hoemig', 'seidl']",Bundesverfassungsgericht - Entscheidungen -,NaN,Bundesverfassungsgericht \n- 1 BvR 410/95 - \n...,https://www.bundesverfassungsgericht.de/Shared...,...,NaN,NaN,NaN,1,29,bundesverfassungsgericht 1 bvr 1788/97 in dem ...,"{'jaeger': ['dm_professions', 'dm_exams', 'dm_...",kuehling,0.30,0.135
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3187,8162,8162,21. September 2022 - 1 BvR 2754/17,'1 BvR 2754/17',1. Senat 3. Kammer,"['britz', 'christ', 'radtke']",Bundesverfassungsgericht - Entscheidungen - ...,Die Beschlüsse des Oberlandesgerichts Celle vo...,BUNDESVERFASSUNGSGERICHT \n\n\n\n\n\n \n\n\n -...,https://www.bundesverfassungsgericht.de/Shared...,...,NaN,NaN,NaN,3,8162,bundesverfassungsgericht 2 bvr 2222/21 in dem ...,"{'huber': ['dm2_municipalities', 'dm2_internat...",kessalwulf,0.37,0.000
3188,8164,8164,27. September 2022 - 1 BvR 2661/21,'1 BvR 2661/21',1. Senat,"['harbarth', 'baer', 'britz', 'ott', 'christ',...",Bundesverfassungsgericht - Entscheidungen - ...,§ 10 Absatz 1 Satz 2 des Gesetzes zur Erhaltun...,Leitsätze \n\n\n\n\n \n\n\nzum Beschluss des E...,https://www.bundesverfassungsgericht.de/Shared...,...,410.0,3.0,dm_taking,4,8164,bundesverfassungsgericht 1 bvr 922/21 in dem v...,"{'britz': ['dm_environmental', 'dm_property', ...",christ,0.23,0.000
3189,8183,8183,21. Oktober 2022 - 2 BvQ 80/22,'2 BvQ 80/22',2. Senat 2. Kammer,"['huber', 'kessalwulf', 'wallrabenstein']",Bundesverfassungsgericht - Entscheidungen - ...,Der Antrag auf Erlass einer einstweiligen Anor...,BUNDESVERFASSUNGSGERICHT \n\n\n\n\n\n \n\n\n -...,https://www.bundesverfassungsgericht.de/Shared...,...,NaN,NaN,NaN,0,8183,bundesverfassungsgericht 2 bvr 2202/19 in dem ...,"{'hermanns': ['dm2_sales', 'dm2_foreclosure', ...",maidowski,0.31,0.000
3190,8185,8185,26. Oktober 2022 - 2 BvE 3/15,"'2 BvE 3/15', '2 BvE 7/15'",2. Senat,"['koenig', 'huber', 'hermanns', 'pmueller', 'k...",Bundesverfassungsgericht - Entscheidungen - ...,Die Verfahren 2 BvE 3/15 und 2 BvE 7/15 werden...,Leitsätze \n\n\n\n\n \n\n\nzum Urteil des Zwei...,https://www.bundesverfassungsgericht.de/Sha

In [17]:
# df_cleaned_final = df_cleaned_final.drop_duplicates('full_text_cleaned').reset_index()

df.memory_usage(index=True, deep=True).sum() / 1000000

204.346211

In [18]:
df['name'].notnull().sum()


117

In [19]:
# get GPT judge predictions, based off domain preds
# domains of judges; align with GPT domain

def get_judge_preds(row):
    gpt_preds = []
    # print(row.domains_of_judges)
    for judge, domains in eval(row.domains_of_judges).items():
        if row['name'] in domains:
            gpt_preds.append(judge)
    return gpt_preds

def get_judge_accuracy(row):
    judge_preds = row['judge_preds']
    # for judge, domains in eval(row.domains_of_judges).items():
    if len(judge_preds) and row['ref_name_final'] and judge_preds[0] == row['ref_name_final']:
        return 1
    else:
        return 0

df['judge_preds'] = df.apply(get_judge_preds, axis=1)
# print(
df['num_preds'] = df['judge_preds'].apply(lambda x: len(x))
print('GPT judge predictions', df['judge_preds'].apply(lambda x: len(x)).value_counts())
df['judge_pred_match'] = df.apply(get_judge_accuracy, axis=1)
df['judge_pred_match'].value_counts()
#   # row.ref_name_final

# df.domains_of_judges
#   print(eval(row.domains_of_judges)[row.ref_name_final])

# df['name']

GPT judge predictions judge_preds
0    3174
1      18
Name: count, dtype: int64


judge_pred_match
0    3188
1       4
Name: count, dtype: int64

In [20]:
df[df.num_preds >0][['judge_preds','ref_name_final']]

,judge_preds,ref_name_final
14,[steiner],grimm
18,[grimm],seidl
328,[jaeger],kuehling
883,[bryde],bryde
1025,[gaier],gaier
1099,[steiner],hoffmannriem
1666,[bryde],schluckebier
1793,[gaier],britz
1888,[baer],schluckebier
1931,[schluckebier],baer


In [34]:
# get keyword method results
# get domains, only the first senate

import numpy as np
domains = np.array([c for c in list(bverfg.columns) if c.startswith('dm_')])
print(domains)
domain_features = df[domains]

# keyword method = 1

df['domains_sum'] = df[domains].sum(axis=1)
# df[domains].sum(axis=1).value_counts()

df[domains]

['dm_environmental' 'dm_property' 'dm_taking' 'dm_farming'
 'dm_restitution' 'dm_postwar' 'dm_ip' 'dm_inheritance' 'dm_art'
 'dm_gaming' 'dm_family' 'dm_names' 'dm_civilstatus' 'dm_transsexual'
 'dm_child' 'dm_parentalallowance' 'dm_custody' 'dm_supportpayment'
 'dm_matriomonial' 'dm_pensionright' 'dm_labour' 'dm_labourleasing'
 'dm_maternityleave' 'dm_corporations' 'dm_association' 'dm_university'
 'dm_science' 'dm_petitions' 'dm_childallowance' 'dm_welfarebasic'
 'dm_asylumallowance' 'dm_welfare' 'dm_vocationalallowance' 'dm_services'
 'dm_professions' 'dm_pensionfund' 'dm_landlord' 'dm_condominium'
 'dm_exams' 'dm_healthinsurance' 'dm_regulation' 'dm_competition'
 'dm_competitionenergy' 'dm_sales' 'dm_attorney' 'dm_tax' 'dm_levies'
 'dm_developmentcharge' 'dm_construction' 'dm_landsale' 'dm_mining'
 'dm_planning' 'dm_landuse' 'dm_conservation' 'dm_landconsolidation'
 'dm_religion' 'dm_school' 'dm_reunification' 'dm_socialsecurity'
 'dm_speech' 'dm_manifestations' 'dm_freedomgeneral'

,dm_environmental,dm_property,dm_taking,dm_farming,dm_restitution,dm_postwar,dm_ip,dm_inheritance,dm_art,dm_gaming,...,dm_religion,dm_school,dm_reunification,dm_socialsecurity,dm_speech,dm_manifestations,dm_freedomgeneral,dm_dataprotection,dm_competitionspeech,dm_torts
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,1,0,1,0,0,0
3,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3187,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3188,1,1,1,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
3189,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3190,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [35]:
row,col = np.where(df[domains]) # indices where = 1
print(row, col)
keyword_method = pd.DataFrame({'row':row, 'col':col, 'domain':domains[col]}
                              ).groupby('row').apply(lambda x: list(x.domain))
df['keyword_method'] = keyword_method
df['keyword_method'] = df['keyword_method'].fillna("")
df['keyword_method']

[   2    2    2 ... 3191 3191 3191] [55 59 61 ... 46 59 61]


/var/folders/b_/djy58vhd1s3d4l09hngvbng40000gp/T/ipykernel_23097/2882452530.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ).groupby('row').apply(lambda x: list(x.domain))


0                                                        
1                                                        
2             [dm_religion, dm_speech, dm_freedomgeneral]
3                                           [dm_property]
4                          [dm_speech, dm_freedomgeneral]
                              ...                        
3187            [dm_property, dm_family, dm_corporations]
3188    [dm_environmental, dm_property, dm_taking, dm_...
3189                                                     
3190                                           [dm_sales]
3191            [dm_levies, dm_speech, dm_freedomgeneral]
Name: keyword_method, Length: 3192, dtype: object

In [23]:
df['keyword_method'].apply(lambda x: len(x)).value_counts()

keyword_method
0     1539
1      749
2      474
3      219
4       88
5       39
9       18
6       13
7       13
8       10
10       7
11       7
16       5
15       4
13       2
12       2
19       1
20       1
21       1
Name: count, dtype: int64

In [24]:
# GPT4 domain pred accuracy, compare to keyword method; accuracy is high

df['keyword_method_len'] = df['keyword_method'].apply(lambda x: len(x))
km1 = df[(df['keyword_method_len'] == 1) & (df['name'].notnull())]
# [df['keyword_method_len'] == 1]
km1.shape, (km1['keyword_method'].apply(lambda x: x[0]) == km1['name']).sum()
# df = df.dropna(subset=['keyword_method'])
# df

((8, 1439), 8)

In [32]:
# get judge predictions, accuracy
# get ground truth judge from KM domain

def get_judge_preds_km(row):
    preds = []
    # print(row.domains_of_judges)
    for km_dm in row['keyword_method']:
        for judge, domains in eval(row.domains_of_judges).items():
            if km_dm in domains:
                preds.append(judge)
    return set(preds)

def get_judge_preds(row, col):
    gpt_preds = []
    # print(row.domains_of_judges)
    for judge, domains in eval(row.domains_of_judges).items():
        if row[col] in domains:
            gpt_preds.append(judge)
    return gpt_preds

def get_judge_accuracy(row, model_pred, ground_truth):
    # judge_preds = row['judge_preds']
    judge_preds = row[model_pred]

    # for judge, domains in eval(row.domains_of_judges).items():
    if len(judge_preds) and row[ground_truth] and judge_preds[0] == row[ground_truth]:
        return 1
    else:
        return 0




# get KM domain
# km1 = df[(df['keyword_method_len'] == 1)]
# km1 = df[(df['keyword_method_len'] == 1) & (df['name'].notnull())]
km1 = df[(df['name'].notnull())]
km1['km1'] = km1['keyword_method'].apply(lambda x: x[0])


# get KM judge predictions, based off domain preds
# km1['km_judge_preds'] = km1.apply(lambda row: get_judge_preds(row, 'km1'), axis=1)
km1['km_judge_preds'] = km1.apply(lambda row: get_judge_preds_km(row), axis=1)
km1['km_judge_preds']

# KM often doesnt predict a judge
km1 = km1[km1['km_judge_preds'].apply(lambda x: len(x)) > 0]

# GPT4 judge accuracy
km1['judge_pred_match'] = km1.apply(lambda row: get_judge_accuracy(row, 'judge_preds', 'km_judge_preds'), axis=1)
print("accuracy of 'judge_preds', 'km_judge_preds'", km1['judge_pred_match'].value_counts())

display(km1[['name', 'keyword_method', 'judge_preds', 'km_judge_preds', 'ref_name_final']])

# 'ref name final' v KM method


# df['num_preds'] = df['judge_preds'].apply(lambda x: len(x))
print(km1['km_judge_preds'].apply(lambda x: len(x)).value_counts())



df['judge_pred_match'] = df.apply(lambda row: get_judge_accuracy(row, 'km_judge_preds', 'ref_name_final'), axis=1)
print('judge_pred_match', df['judge_pred_match'].value_counts())

# df.ref_name_final # is this accurate?



# # train LR to predict domain
# # prediction target: random KM domain candidate

# # features: 
# # use keyword method results, GPT4: chamber+senate 1


# # # calculate domain scores for cases, using the ref and judge domains
# # # model a case and domain scores

# # # for a new case, predict domains, judge-domain score, referee
# # # features: judges, judge domains, domain scores, "judge scores"


accuracy of 'judge_preds', 'km_judge_preds' judge_pred_match
0    48
Name: count, dtype: int64


/var/folders/b_/djy58vhd1s3d4l09hngvbng40000gp/T/ipykernel_23097/10227861.py:38: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  km1['km1'] = km1['keyword_method'].apply(lambda x: x[0])
/var/folders/b_/djy58vhd1s3d4l09hngvbng40000gp/T/ipykernel_23097/10227861.py:43: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  km1['km_judge_preds'] = km1.apply(lambda row: get_judge_preds_km(row), axis=1)


,name,keyword_method,judge_preds,km_judge_preds,ref_name_final
14,dm_socialsecurity,"[dm_property, dm_labour, dm_childallowance, dm...",[steiner],"{haas, grimm, kuehling, jaeger, steiner, seidl}",grimm
18,dm_speech,"[dm_religion, dm_speech, dm_freedomgeneral]",[grimm],"{hoemig, grimm}",seidl
34,dm_healthinsurance,"[dm_property, dm_family, dm_professions, dm_he...",[],"{papier, steiner}",haas
49,dm_professions,"[dm_ip, dm_inheritance, dm_association, dm_pro...",[],"{papier, grimm}",papier
97,dm_university,"[dm_environmental, dm_property, dm_university,...",[],"{papier, haas}",papier
243,dm_ip,"[dm_property, dm_ip, dm_family, dm_professions...",[],{jaeger},jaeger
268,dm_family,"[dm_environmental, dm_property, dm_taking, dm_...",[],{papier},papier
328,dm_professions,"[dm_university, dm_professions]",[jaeger],"{jaeger, kuehling}",kuehling
453,dm_reunification,"[dm_property, dm_reunification]",[],{papier},hoffmannriem
883,dm_labour,"[dm_property, dm_family, dm_labour, dm_associa...",[bryde],"{jaeger, bryde}",bryde


km_judge_preds
1    21
2    20
3     5
6     1
4     1
Name: count, dtype: int64


KeyError: 'km_judge_preds'

In [ ]:
def get_judge_preds(row):
    gpt_preds = []
    # print(row.domains_of_judges)
    for judge, domains in eval(row.domains_of_judges).items():
        if row['name'] in domains:
            gpt_preds.append(judge)
    return gpt_preds

def get_judge_accuracy(row):
    judge_preds = row['judge_preds']
    # for judge, domains in eval(row.domains_of_judges).items():
    if len(judge_preds) and row['ref_name_final'] and judge_preds[0] == row['ref_name_final']:
        return 1
    else:
        return 0

df['judge_preds'] = df.apply(get_judge_preds, axis=1)

In [ ]:
df['km1']

KeyError: 'km1'

## Prediction target

In [ ]:
# prediction target: random KM domain candidate
# TODO: try GPT4 predictions; use keyword_method output as features

# df['domain'] = df['keyword_method'].apply(lambda x: x[0] if len(x) > 0 else None)
df['domain'] = df['name']

# gpt_predictions = bverfg[bverfg['name'].notnull()]


In [ ]:
list(df.columns)

['Unnamed: 0',
 'id',
 'date_and_first_docket',
 'az',
 'senate_and_chamber',
 'judges',
 'short_description',
 'decision',
 'full_text',
 'url',
 'date',
 'year',
 'casecomplaint',
 'caseelect',
 'caseprelim',
 'casetype',
 'justbaer',
 'justbritz',
 'justbross',
 'justbryde',
 'justchrist',
 'justdifabio',
 'justeichberger',
 'justfkirchhof',
 'justgaier',
 'justgerhardt',
 'justgrasshof',
 'justgrimm',
 'justhaas',
 'justhaertel',
 'justharbarth',
 'justhassemer',
 'justhermanns',
 'justhoemig',
 'justhohmanndennhardt',
 'justhoffmannriem',
 'justhuber',
 'justjaeger',
 'justjentsch',
 'justkessalwulf',
 'justkoenig',
 'justkruis',
 'justkuehling',
 'justlandau',
 'justlangenfeld',
 'justlimbach',
 'justluebbewolff',
 'justmaidowski',
 'justmasing',
 'justmellinghoff',
 'justosterloh',
 'justott',
 'justpapier',
 'justpaulus',
 'justpkirchhof',
 'justpmueller',
 'justradtke',
 'justschluckebier',
 'justseibert',
 'justseidl',
 'justsommer',
 'juststeiner',
 'justvosskuhle',
 'justwa

In [ ]:
# case embeddings map

# torch.save(document_embeddings, os.path.join(root, 'data/case_embeddings_avg_map.pt'))
document_embeddings_map = torch.load(os.path.join(root, 'data/case_embeddings_avg_map.pt'))
print(len(document_embeddings_map))


# df_cleaned_final['case_embedding']
document_embeddings = df_cleaned_final['full_text_cleaned'].apply(lambda doc: document_embeddings_map[doc])
document_embeddings_avg = document_embeddings


document_embeddings_avg_numpy = [tensor.detach().cpu().numpy() for tensor in document_embeddings_avg]

numeric_array = np.array(df_cleaned_final[['prob','combined_domains_score']])

len(document_embeddings_avg_numpy)

3188


3192

In [ ]:
# df_cleaned_final.judges
df.columns

NameError: name 'df' is not defined

In [ ]:
# judge candidate and judge domain features

import sklearn
from sklearn.feature_extraction import DictVectorizer
from sklearn.preprocessing import LabelEncoder

def encode_judges(judges):
    try:
        return dict(zip(eval(judges), [1,1,1]))
    except:
        print('error', judges)
        return {}

df['judges'] = df.judges.apply(encode_judges)
    # lambda l: dict(zip(eval(l), [1,1,1])))

dv = DictVectorizer(sparse=False)
D = df.judges
domain_features = dv.fit_transform(D)

le = LabelEncoder()
le.fit(df.ref_name_final)
# le.classes_
df_cleaned_final['target'] = le.transform(df.ref_name_final)
# le.inverse_transform([0, 0, 1, 2])

df_cleaned_final['target']

# encode judge domains


error ['seidl', 'grimm', 'kuehling', 'jaeger', 'haas', 'hoemig',  , 'steiner',  ]
error ['seidl', 'grimm', 'kuehling', 'jaeger', 'haas', 'hoemig',  , 'steiner',  ]
error [, 'grimm', 'hoemig']
error ['papier', 'grimm', 'kuehling', 'jaeger', 'haas', 'hoemig',  , 'steiner',  ]
error ['papier', 'grimm', 'kuehling',  ,  ,  , 'jaeger', 'haas', 'hoemig',  ,  , 'steiner']
error ['papier', 'grimm', 'kuehling', 'jaeger', 'haas', 'hoemig', 'steiner',  , 'hohmanndennhardt']
error ['papier', 'kuehling', 'jaeger', 'haas', 'hoemig', 'steiner',  , 'hohmanndennhardt', 'hoffmannriem']
error ['papier', 'kuehling', 'jaeger', 'haas', 'hoemig', 'steiner',  , 'hohmanndennhardt', 'hoffmannriem']
error ['limbach', 'sommer', 'jentsch',  , 'hassemer', 'brossss', 'osterloh', 'difabio', 'mellinghoff']
error ['papier', 'jaeger', 'hoemig', 'steiner',  , 'hohmanndennhardt',  , 'hoffmannriem',  ]
error ['papier', 'jaeger', 'haas', 'hoemig', 'steiner', 'hohmanndennhardt', 'hoffmannriem',  , 'bryde']
error ['papier', 'j

AttributeError: 'DataFrame' object has no attribute 'ref_name_final'

In [ ]:
# # domain encoder

# def get_labels(d):
#   res = {}
#   for k,domains in d.items():
#     for d in domains:
#       res[f'{k}_{d}'] = 1
#   return res

# # judge_domains = df.domains_of_judges.apply(lambda d: get_labels(eval(d)))
# judge_domains = df.domains_of_judges.apply(lambda d: get_labels(eval(d)))


# domain_vec = DictVectorizer(sparse=False)
# encoded_domains = domain_vec.fit_transform(judge_domains)
# encoded_domains.shape

In [ ]:
# dv.get_feature_names_out()

# document_embeddings_avg_numpy_updated = np.concatenate([domain_features, numeric_array, document_embeddings_avg_numpy],axis=1)
# doc_embedding_features = np.concatenate([domain_features, document_embeddings_avg_numpy],axis=1)
doc_embedding_features = np.concatenate([domain_features, document_embeddings_avg_numpy],axis=1)

In [ ]:
# TFIDF features

import scipy

from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer()
cases_tfidf = vectorizer.fit_transform(df_cleaned_final['full_text_cleaned'])
# vectorizer.get_feature_names_out()
print(cases_tfidf.shape)


from scipy.sparse import hstack
# tfidf_features = hstack([scipy.sparse.csr_matrix(domain_features), cases_tfidf, scipy.sparse.csr_matrix(numeric_array)])
tfidf_features = hstack([scipy.sparse.csr_matrix(domain_features), cases_tfidf])


(3192, 118826)


In [ ]:
# topic model features

selected_features_all = (df.columns.str.contains('prob'))|(df.columns=='combined_domains_score')
domain_features = (df.columns.str.contains('prob'))&(df.columns.str.contains('dm'))
# selected_features_all = (train_dataset.columns=='prob')| train_dataset.columns.str.contains('one_case_one_judge_domain_score')|\
#                         (train_dataset.columns=='combined_domains_score')
# judge_domain_score
# selected_features_no_prob = train_dataset.columns.str.contains('one_case_one_judge_domain_score')|\
#                         (train_dataset.columns=='combined_domains_score')
# selected_features_no_combined_domains_score = (train_dataset.columns=='prob')|train_dataset.columns.str.contains('one_case_one_judge_domain_score')
# selected_features_no_one_case_one_judge_domain_score = (train_dataset.columns=='prob')| (train_dataset.columns=='combined_domains_score')



# tm_features = df.iloc[:,selected_features_all].to_numpy()
# tm_features = np.concatenate((tm_features, domain_features), axis=1)
# tm_features.shape

# # selected_features_all
# topic_model_feature_names = df.iloc[:,selected_features_all].columns
# topic_model_feature_names

In [ ]:

# get train-test split
# If we split temporally then we need to handle unseen/rare judges in the test set

# train_size = 0.8
# train_dataset = df_cleaned_final.sample(frac=train_size,random_state=200)
# test_dataset = df_cleaned_final.drop(train_dataset.index).reset_index(drop=True)
# train_dataset = train_dataset.reset_index(drop=True)

# train_dataset = df_cleaned_final[df_cleaned_final['year']<=2017]
train_dataset = df_cleaned_final[df_cleaned_final['year']<=2020]
test_dataset = df_cleaned_final.drop(train_dataset.index)
# train_dataset = train_dataset.reset_index(drop=True)

print("FULL Dataset: {}".format(df_cleaned_final.shape))
print("TRAIN Dataset: {}".format(train_dataset.shape))
print("TEST Dataset: {}".format(test_dataset.shape))


FULL Dataset: (3192, 1434)
TRAIN Dataset: (3016, 1434)
TEST Dataset: (176, 1434)


In [ ]:
# check there is no train-test overlap
set(test_dataset.full_text_cleaned) & set(train_dataset.full_text_cleaned)

set()

In [ ]:
# # Drop test examples with judges that were rarely seen in train

# # unseen_judges = set(test_dataset.ref_name_final) - set(train_dataset.ref_name_final)
# keep_test_judges = []
# for judge in set(test_dataset.ref_name_final):
#   train_cnt = (train_dataset.ref_name_final == judge).sum()
#   if train_cnt > 2:
#     keep_test_judges.append(judge)
#   print(judge, train_cnt)


In [ ]:

train_index = train_dataset.index
test_index = test_dataset.index

In [ ]:
%%time

# run train-test prediction experiments
# random guessing gets?

from sklearn.linear_model import LogisticRegression
import sklearn.metrics as mx
from sklearn.metrics import classification_report, accuracy_score

import numpy as np
from sklearn.dummy import DummyClassifier
from joblib import dump, load


def run_prediction_experiment(X, y, save_name):

    # X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(
    #   *[X, y], test_size=0.2, train_size=None, random_state=42, shuffle=True, stratify=None)

    # custom temporal split, to prevent lookahead bias
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # model = LogisticRegression(fit_intercept=True,max_iter = 100000)
    model = LogisticRegression()
    # model = LogisticRegression(solver = 'sag', random_state = 1)

    model.fit(X_train,y_train)
    y_pred = model.predict(X_test)
    # y_pred_prob = logmodel.predict_proba(X_test)[:,1]
    print(classification_report(y_test,y_pred))

    dump(model, os.path.join(root, f'{save_name}.joblib'))
    # # load(model, 'tfidf_logreg.joblib')
    return model

# TODO: custom test set
def run_train_test_experiment(X, y, train_index, test_index, save_name):

    # X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(
    #   *[X, y], test_size=0.2, train_size=None, random_state=42, shuffle=True, stratify=None)

    # custom temporal split, to prevent lookahead bias
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    model = LogisticRegression(fit_intercept=True,max_iter = 100000)
    # model = LogisticRegression()
    # model = LogisticRegression(solver = 'sag', random_state = 1)

    model.fit(X_train,y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test,y_pred)
    print(classification_report(y_test,y_pred))

    dump(model, os.path.join(root, f'{save_name}.joblib'))
    # # load(model, 'tfidf_logreg.joblib')
    return acc


def run_train_test_experiment_trials(X, y, save_name):

    # custom temporal split, to prevent lookahead bias
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    accuracy_scores = []

    for random_state in [1, 2, 3]:
      # model = LogisticRegression(fit_intercept=True,max_iter = 100000)
      model = LogisticRegression(solver = 'sag', random_state = random_state)
      model.fit(X_train,y_train)
      y_pred = model.predict(X_test)
      accuracy = accuracy_score(y_test,y_pred)
      accuracy_scores.append(accuracy)
      print(random_state, accuracy)

    print(accuracy_scores, sum(accuracy_scores)/len(accuracy_scores))
    dump(model, os.path.join(root, f'{save_name}.joblib'))
    return model

CPU times: user 23.2 ms, sys: 1.57 ms, total: 24.8 ms
Wall time: 39.8 ms


In [ ]:
# time series cross val

def get_train_test_split(df, year):

  train_dataset = df[df['year']<=year]
  # test_dataset = df_cleaned_final.drop(train_dataset.index)
  test_dataset = df[((df['year']>year) & (df['year']<year+3))]

  # Drop test examples with judges that are rarely seen in train
  # keep_test_judges = []
  # for judge in set(test_dataset.ref_name_final):
  #   train_cnt = (train_dataset.ref_name_final == judge).sum()
  #   if train_cnt > 2:
  #     keep_test_judges.append(judge)
  #     print(judge, train_cnt)
  # test_dataset = test_dataset[test_dataset.ref_name_final.isin(keep_test_judges)]
  # print(len(test_dataset))

  train_index = train_dataset.index
  test_index = test_dataset.index

  return train_index, test_index


def get_time_series_cv_results():
  return

acc_scores = []
for year in [2005, 2010, 2017, 2020]:

  # train_dataset = df[df['year']<=year]
  # # test_dataset = df_cleaned_final.drop(train_dataset.index)
  # test_dataset = df[((df['year']>year) & (df['year']<year+3))]

  # # Drop test examples with judges that are rarely seen in train
  # keep_test_judges = []
  # for judge in set(test_dataset.ref_name_final):
  #   train_cnt = (train_dataset.ref_name_final == judge).sum()
  #   if train_cnt > 2:
  #     keep_test_judges.append(judge)
  #     print(judge, train_cnt)
  # test_dataset = test_dataset[test_dataset.ref_name_final.isin(keep_test_judges)]
  # print(len(test_dataset))

  # train_index = train_dataset.index
  # test_index = test_dataset.index
  train_index, test_index = get_train_test_split(df, year)

  # run_train_test_experiment(X, y, train_index, test_index, save_name)

  # acc = run_train_test_experiment(
  #   tfidf_features,
  #   df.ref_name_final,
  #   train_index,
  #   test_index,
  #   "tfidf"
  # )
  # acc = run_train_test_experiment(
  #   tm_features,
  #   df.ref_name_final,
  #   train_index,
  #   test_index,
  #   "topic_model"
  # )
  # .74, .6, .72, .62
  acc = run_train_test_experiment(
    doc_embedding_features,
    df.domain,
    train_index,
    test_index,
    "doc_embeddings_avg"
  )
  # acc = run_train_test_experiment(
  #   np.concatenate([tm_features, doc_embedding_features,encoded_domains], axis=1),
  #   df.ref_name_final,
  #   train_index,
  #   test_index,
  #   "embedding+topic_model+domains"
  # )
  acc_scores.append(acc)

print(acc_scores)
sum(acc_scores)/len(acc_scores)

ValueError: Input contains NaN

In [ ]:
# !pip install deep_translator

# from deep_translator import GoogleTranslator

# # Use any translator you like, in this example GoogleTranslator
# translator = GoogleTranslator(source='de', target='en')

# translated = []
# for x in df_cleaned_final['full_text_cleaned']:
#   try:
#     translated.append(translator.translate(x))  # output -> Weiter so, du bist großartig
#   except Exception as e:
#     print(e)

# translated
# len(translated)

# import pickle

# with open(os.path.join(root, 'data/translated_cases.pkl'), 'rb') as f:
#   trans = pickle.load(f)

# trans = list(trans.items())
# len(trans)

In [ ]:
train_index, test_index = get_train_test_split(df, 2017)


In [ ]:
run_prediction_experiment(
    encoded_judges,
    df.ref_name_final,
    "judges"
)

NameError: name 'encoded_judges' is not defined

In [ ]:
%%time

run_prediction_experiment(
    tm_features,
    df.ref_name_final,
    "ATM_features"
)

              precision    recall  f1-score   support

        baer       0.71      0.94      0.81        18
       britz       0.80      0.67      0.73         6
  eichberger       0.00      0.00      0.00         4
   fkirchhof       1.00      1.00      1.00        19
    hermanns       0.00      0.00      0.00         3
       huber       0.33      0.33      0.33         6
  kessalwulf       0.54      0.73      0.62        26
      koenig       0.50      0.07      0.12        14
  langenfeld       0.73      0.92      0.81        12
   maidowski       0.82      0.95      0.88        19
      masing       0.66      1.00      0.79        19
      paulus       0.00      0.00      0.00        11
    pmueller       0.50      0.50      0.50         6
   vosskuhle       1.00      1.00      1.00         1

    accuracy                           0.70       164
   macro avg       0.54      0.58      0.54       164
weighted avg       0.61      0.70      0.63       164

CPU times: user 922 ms, s

/home/amao/miniconda3/envs/nlp/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/amao/miniconda3/envs/nlp/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/amao/miniconda3/envs/nlp/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", le

LogisticRegression()

In [ ]:
%%time

# embeddings alone

run_prediction_experiment(
    doc_embedding_features,
    df.ref_name_final,
    "doc_embeddings_avg"
)

              precision    recall  f1-score   support

        baer       0.89      0.94      0.92        18
       britz       0.67      0.33      0.44         6
  eichberger       0.57      1.00      0.73         4
   fkirchhof       1.00      1.00      1.00        19
    hermanns       0.50      0.33      0.40         3
       huber       0.27      0.50      0.35         6
  kessalwulf       0.56      0.54      0.55        26
      koenig       0.64      0.50      0.56        14
  langenfeld       0.70      0.58      0.64        12
   maidowski       0.86      0.95      0.90        19
      masing       0.78      0.95      0.86        19
      paulus       0.83      0.45      0.59        11
    pmueller       0.40      0.33      0.36         6
   vosskuhle       0.50      1.00      0.67         1

    accuracy                           0.72       164
   macro avg       0.66      0.67      0.64       164
weighted avg       0.73      0.72      0.71       164

CPU times: user 4.77 s, s

/home/amao/miniconda3/envs/nlp/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

In [ ]:
%%time

run_prediction_experiment(
    np.concatenate([tm_features, doc_embedding_features], axis=1),
    df.ref_name_final,
    "embedding+topic_model"
)

# run_train_test_experiment_trials(
#     np.concatenate([tm_features, doc_embedding_features], axis=1),
#     df.ref_name_final,
#     "embedding+topic_model"
# )

              precision    recall  f1-score   support

        baer       0.94      0.94      0.94        18
       britz       0.71      0.83      0.77         6
  eichberger       0.75      0.75      0.75         4
   fkirchhof       1.00      1.00      1.00        19
    hermanns       1.00      0.33      0.50         3
       huber       0.22      0.33      0.27         6
  kessalwulf       0.61      0.65      0.63        26
      koenig       0.70      0.50      0.58        14
  langenfeld       0.73      0.92      0.81        12
   maidowski       0.82      0.95      0.88        19
      masing       0.73      1.00      0.84        19
      paulus       1.00      0.27      0.43        11
    pmueller       1.00      0.17      0.29         6
   vosskuhle       1.00      1.00      1.00         1

    accuracy                           0.76       164
   macro avg       0.80      0.69      0.69       164
weighted avg       0.79      0.76      0.74       164

CPU times: user 5.94 s, s

/home/amao/miniconda3/envs/nlp/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

In [ ]:
%%time

run_prediction_experiment(
    tfidf_features,
    df['target'],
    "tfidf"
)

              precision    recall  f1-score   support

           0       0.75      1.00      0.86        18
           1       1.00      0.67      0.80         6
           6       1.00      0.25      0.40         4
           7       1.00      1.00      1.00        19
          16       1.00      0.33      0.50         3
          20       0.00      0.00      0.00         6
          23       0.60      0.81      0.69        26
          24       1.00      0.36      0.53        14
          28       0.69      0.75      0.72        12
          31       0.78      0.95      0.86        19
          32       0.66      1.00      0.79        19
          37       0.00      0.00      0.00        11
          39       0.38      0.50      0.43         6
          45       0.00      0.00      0.00         1

    accuracy                           0.72       164
   macro avg       0.63      0.54      0.54       164
weighted avg       0.69      0.72      0.67       164

CPU times: user 3min 14s,

/home/amao/miniconda3/envs/nlp/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/amao/miniconda3/envs/nlp/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/amao/miniconda3/envs/nlp/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", le

LogisticRegression()

In [ ]:
%%time

run_prediction_experiment(
    hstack([scipy.sparse.csr_matrix(encoded_judges), cases_tfidf, scipy.sparse.csr_matrix(tm_features)]),

    df['target'],
    "tfidf_tm"
)

              precision    recall  f1-score   support

           0       0.71      0.94      0.81        18
           1       1.00      0.67      0.80         6
           6       0.00      0.00      0.00         4
           7       1.00      1.00      1.00        19
          16       0.00      0.00      0.00         3
          20       0.50      0.33      0.40         6
          23       0.64      0.81      0.71        26
          24       1.00      0.43      0.60        14
          28       0.73      0.92      0.81        12
          31       0.82      0.95      0.88        19
          32       0.66      1.00      0.79        19
          37       0.00      0.00      0.00        11
          39       0.50      0.50      0.50         6
          45       1.00      1.00      1.00         1

    accuracy                           0.74       164
   macro avg       0.61      0.61      0.59       164
weighted avg       0.68      0.74      0.69       164

CPU times: user 3min 56s,

/home/amao/miniconda3/envs/nlp/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/amao/miniconda3/envs/nlp/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/amao/miniconda3/envs/nlp/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", le

LogisticRegression()

In [ ]:
# model = load('ATM_features.joblib')
model = load(os.path.join(root, f'tfidf_tm.joblib'))
# dump(model, os.path.join(root, f'ATM_features.joblib'))

model

LogisticRegression()

In [ ]:
# feature importance

# Get the coefficients (weights) of the features
coefficients = [round(x,3) for x in model.coef_[0]]

# Get the names of the features
# hstack([scipy.sparse.csr_matrix(encoded_judges), cases_tfidf, scipy.sparse.csr_matrix(tm_features)]),
# feature_names = np.concatenate([dv.get_feature_names_out(), vectorizer.get_feature_names_out()])
feature_names = np.concatenate([dv.get_feature_names_out(), vectorizer.get_feature_names_out(), topic_model_feature_names])
assert len(feature_names) == len(coefficients)

# Create a dictionary mapping feature names to their corresponding coefficients
feature_coefficients = dict(zip(feature_names, coefficients))

# Print the feature names and their coefficients, sorted by coefficient magnitude
sorted_features = sorted(feature_coefficients.items(), key=lambda x: abs(x[1]), reverse=True)


AssertionError: 

In [ ]:
len(feature_names), len(coefficients)

: 

In [ ]:
sorted_features[:20]

: 

In [ ]:
# len(' '.join(feature_names))
top_feature_names = list(zip(*sorted_features))[0][:100]
# ' '.join(
# len(top_feature_names)

top_feature_names[:50]

: 

In [ ]:
# 'fkirchhof',
#  'bundesarbeitsgericht',
#  'landesarbeitsgerichts',
#  'bundesarbeitsgerichts',
#  'marburg',
#  'landesarbeitsgericht',
#  'gieen',
#  'beratungshilfe',
#  'universitt',
#  'der',
#  'sgb',
#  'as',
#  'tkg',
#  'ukg',
#  'sa',
#  'arbeitsgerichts',


# 'fkirchhof',
#  'federal labor court',
#  'state labor court',
#  'federal labor court',
#  'marburg',
#  'state labor court',
#  'pour',
#  'advisory assistance',
#  'university',
#  'the',
#  'sgb',
#  'as',
#  'tkg',
#  'ukg',
#  'sa',
#  'labor court',

: 

In [ ]:
# # !pip install deep_translator

# from deep_translator import GoogleTranslator

# # Use any translator you like, in this example GoogleTranslator
# translator = GoogleTranslator(source='de', target='en')

# translated = []
# for x in top_feature_names:
#   translated.append(translator.translate(x))  # output -> Weiter so, du bist großartig

# translated

: 

In [ ]:
# analyze failures

# list(df.columns)
df[df.ref_name_final == 'christ']

: 

In [ ]:
df

: 

: 

: 

In [ ]:
document_embeddings_avg_numpy_train = document_embeddings_avg_numpy[0:len(training_set_logistic['juid'])]
document_embeddings_avg_numpy_test = document_embeddings_avg_numpy[len(training_set_logistic['juid']):len(document_embeddings_avg_numpy)]
# true_label_train = df_cleaned_final[['ref_in_judge']][0:len(training_set_logistic['juid'])]
# true_label_test = df_cleaned_final[['ref_in_judge']][len(training_set_logistic['juid']):len(document_embeddings_avg_numpy)]
true_label_train = df_cleaned_final[['target']][0:len(training_set_logistic['juid'])]
true_label_test = df_cleaned_final[['target']][len(training_set_logistic['juid']):len(document_embeddings_avg_numpy)]


: 

In [ ]:
document_embeddings_avg_numpy_train_updated = document_embeddings_avg_numpy_updated[0:len(training_set_logistic['juid'])]
document_embeddings_avg_numpy_test_updated = document_embeddings_avg_numpy_updated[len(training_set_logistic['juid']):len(document_embeddings_avg_numpy_updated)]


: 

In [ ]:
training_set_logistic.clean_judges

: 

In [ ]:
training_set_logistic['embeddings'] = document_embeddings_avg_numpy_train

: 

In [ ]:
# Convert time to datetime
training_set_logistic['date'] = pd.to_datetime(training_set_logistic['date'])

# Group by author and calculate the average embedding before each time
training_embeddings_adjusted = training_set_logistic.groupby(['type', 'date']).apply(lambda x: x[x['date'] <= x['date'].max()]['embeddings'].sum()).reset_index()
training_embeddings_adjusted.columns = ['type','date','embeddings_adjusted']

: 

In [ ]:
training_set_logistic = pd.merge(training_set_logistic,training_embeddings_adjusted, how = 'left', on = ['type','date'])

: 

In [ ]:
testing_set_logistic['embeddings'] = document_embeddings_avg_numpy_test
# Convert time to datetime
testing_set_logistic['date'] = pd.to_datetime(testing_set_logistic['date'])

# Group by author and calculate the average embedding before each time
testing_embeddings_adjusted = testing_set_logistic.groupby(['type', 'date']).apply(lambda x: x[x['date'] <= x['date'].max()]['embeddings'].sum()).reset_index()
testing_embeddings_adjusted.columns = ['type','date','embeddings_adjusted']

: 

In [ ]:
testing_set_logistic = pd.merge(testing_set_logistic,testing_embeddings_adjusted, how = 'left', on = ['type','date'])

: 

In [ ]:
training_set_logistic

: 

In [ ]:
training_embeddings = np.array(training_set_logistic['embeddings_adjusted'].values.tolist())
testing_embeddings = np.array(testing_set_logistic['embeddings_adjusted'].values.tolist())

: 

In [ ]:
len(training_embeddings)

: 

In [ ]:
# from sklearn.linear_model import LogisticRegression
# logmodel = LogisticRegression(fit_intercept=True,max_iter = 100000)
# logmodel.fit(document_embeddings_avg_numpy_train,true_label_train)
# y_pred = logmodel.predict(document_embeddings_avg_numpy_test)
# from sklearn.metrics import classification_report
# print(classification_report(true_label_test,y_pred))

: 

In [ ]:
# sum(y_pred)

: 

In [ ]:
# from sklearn.linear_model import LogisticRegression
# logmodel = LogisticRegression(fit_intercept=True,max_iter = 100000)
# logmodel.fit(training_embeddings,true_label_train)
# y_pred = logmodel.predict(testing_embeddings)
# from sklearn.metrics import classification_report
# print(classification_report(true_label_test,y_pred))

: 

In [ ]:
sum(y_pred)

: 

In [ ]:
from sklearn.linear_model import LogisticRegression
logmodel = LogisticRegression(fit_intercept=True,max_iter = 100000)
logmodel.fit(document_embeddings_avg_numpy_train_updated,true_label_train)
y_pred = logmodel.predict(document_embeddings_avg_numpy_test_updated)
y_pred_prob = logmodel.predict_proba(document_embeddings_avg_numpy_test_updated)[:,1]
from sklearn.metrics import classification_report
print(classification_report(true_label_test,y_pred))

: 

In [ ]:
def compute_case_accuracy(test_dataset,y_pred,y_pred_prob):
  test_metric = test_dataset[['uid','ref_in_judge']]
  test_metric['y_pred'] = y_pred
  test_metric['y_pred_prob'] = y_pred_prob
  test_metric['y_pred_nmax'] = (
      test_metric
      .groupby('uid')['y_pred_prob']
      .transform(
          lambda v: v.eq(v.max())
      )
      .astype(int)
  )
  print(test_metric)
  case_arr_naive = test_metric[test_metric['ref_in_judge']==1]['y_pred'].sum()/len(np.unique(test_metric.uid))
  case_arr = test_metric[test_metric['ref_in_judge']==1]['y_pred_nmax'].sum()/len(np.unique(test_metric.uid))
  print('case_arr_navie: ', case_arr_naive)
  print('case_arr: ', case_arr)
  return test_metric, case_arr_naive, case_arr

: 

In [ ]:
test_metric, case_arr_naive, case_arr = compute_case_accuracy(testing_set_logistic,y_pred,y_pred_prob)

: 

In [ ]:
training_embeddings_updated = np.concatenate([training_embeddings,numeric_array[0:len(training_set_logistic['juid'])]],axis=1)
testing_embeddings_updated = np.concatenate([testing_embeddings,numeric_array[len(training_set_logistic['juid']):]],axis=1)

: 

In [ ]:
from sklearn.linear_model import LogisticRegression
logmodel = LogisticRegression(fit_intercept=True,max_iter = 100000)
logmodel.fit(training_embeddings_updated,true_label_train)
y_pred = logmodel.predict(testing_embeddings_updated)
y_pred_prob = logmodel.predict_proba(testing_embeddings_updated)[:,1]
from sklearn.metrics import classification_report
print(classification_report(true_label_test,y_pred))

: 

In [ ]:
test_metric, case_arr_naive, case_arr = compute_case_accuracy(testing_set_logistic,y_pred,y_pred_prob)

: 

In [ ]:
df_cleaned_final['embeddings'] = document_embeddings_avg_numpy
# Convert time to datetime
df_cleaned_final['date'] = pd.to_datetime(df_cleaned_final['date'])

# Group by author and calculate the average embedding before each time
df_embeddings_adjusted = df_cleaned_final.sort_values(['type', 'date']).groupby(['type', 'date']).apply(lambda x: x[x['date'] <= x['date'].max()]['embeddings'].sum()).reset_index()
df_embeddings_adjusted.columns = ['type','date','embeddings_adjusted']

df_cleaned_final_all = pd.merge(df_cleaned_final,df_embeddings_adjusted, how = 'left', on = ['type','date'])

: 

In [ ]:
df_embeddings_judge = np.array(df_cleaned_final_all['embeddings_adjusted'].values.tolist())

: 

In [ ]:
df_embeddings_judge

: 

In [ ]:
training_embedding_modify = df_embeddings_judge[0:len(training_set_logistic['juid'])]
testing_embedding_modify= df_embeddings_judge[len(training_set_logistic['juid']):len(document_embeddings_avg_numpy)]

: 

In [ ]:
from sklearn.linear_model import LogisticRegression
logmodel = LogisticRegression(fit_intercept=True,max_iter = 100000)
logmodel.fit(training_embedding_modify,true_label_train)
y_pred = logmodel.predict(testing_embedding_modify)
from sklearn.metrics import classification_report
print(classification_report(true_label_test,y_pred))

: 

## Binary Prediction With ATM Features

In [ ]:
# list of legal documents
legal_documents = []

# The labels for the samples.
labels = []

selected_features_domain_scores = df_cleaned_final.columns.str.contains('one_case_one_judge_domain_score')

for index, row in df_cleaned_final.iterrows():
    # Add the combined text to the list.
    combined_text = ''
    combined_text += '[SEP]'.join(row["clean_judges"])
    combined_text += "[SEP] {:} Author Probability [SEP] {:} ".format(row['clean_judges_final'],row["prob"])
    for i in range(df_cleaned_final.iloc[:,selected_features_domain_scores].shape[1]):
        column_name = df_cleaned_final.iloc[:,selected_features_domain_scores].columns[i]
        column_value = df_cleaned_final.iloc[index,selected_features_domain_scores][i]
        combined_text += "[SEP]"+ column_name+"[SEP]"+column_value.astype('str')
    combined_text += "[SEP] Combined Domain Scores [SEP] {:} ".format(row["combined_domains_score"])
    combined_text += row['full_text_cleaned']
    legal_documents.append(combined_text)

    # Also record the sample's label.
    labels.append(row["ref_in_judge"])

: 

In [ ]:
from transformers import BertTokenizer

# Load the BERT tokenizer.
print('Loading BERT tokenizer...')
tokenizer = BertTokenizer.from_pretrained('dbmdz/bert-base-german-cased', do_lower_case=True)

: 

In [ ]:
from transformers import BertForSequenceClassification

# Load BertForSequenceClassification, the pretrained dbmdz/bert-base-german-cased BERT model
model = BertForSequenceClassification.from_pretrained(
    "dbmdz/bert-base-german-cased", # Use the dbmdz/bert-base-german-cased BERT model, with an cased vocab.
    num_labels = 2, # The number of output labels--2 for binary classification.
)

# Tell pytorch to run this model on the GPU.
desc = model.cuda()

: 

In [ ]:
# Larger batch sizes tend to be better, and we can fit this in memory.
batch_size = 32

# set learning rate
learning_rate = 1e-5

# Number of training epochs.
epochs = 4

# max length
max_len = 200

: 

In [ ]:
# Tokenize all of the sentences and map the tokens to thier word IDs.
input_ids = []
attention_masks = []

for doc in legal_documents:
    # `encode_plus` will:
    #   (1) Tokenize the sentence.
    #   (2) Prepend the `[CLS]` token to the start.
    #   (3) Append the `[SEP]` token to the end.
    #   (4) Map tokens to their IDs.
    #   (5) Pad or truncate the sentence to `max_length`
    #   (6) Create attention masks for [PAD] tokens.
    encoded_dict = tokenizer.encode_plus(
                        doc,                      # Sentence to encode.
                        add_special_tokens = True, # Add '[CLS]' and '[SEP]'
                        max_length = max_len,           # Pad & truncate all sentences.
                        truncation = True,
                        padding = 'max_length',
                        return_attention_mask = True,   # Construct attn. masks.
                        return_tensors = 'pt',     # Return pytorch tensors.
                )

    # Add the encoded sentence to the list.
    input_ids.append(encoded_dict['input_ids'])

    # And its attention mask (simply differentiates padding from non-padding).
    attention_masks.append(encoded_dict['attention_mask'])

# Convert the lists into tensors.
input_ids = torch.cat(input_ids, dim=0)
attention_masks = torch.cat(attention_masks, dim=0)
labels = torch.tensor(labels)

print('DONE.')

: 

In [ ]:
training_set_logistic = pd.read_csv('data/input/training_set_logistic.csv')
testing_set_logistic = pd.read_csv('data/input/testing_set_logistic.csv')

: 

In [ ]:
train_idx = training_set_logistic['juid'].values
test_idx = testing_set_logistic['juid'].values

: 

In [ ]:
from torch.utils.data import TensorDataset

# Split the samples, and create TensorDatasets for each split.
train_dataset = TensorDataset(input_ids[train_idx], attention_masks[train_idx], labels[train_idx])
test_dataset = TensorDataset(input_ids[test_idx], attention_masks[test_idx], labels[test_idx])

: 

In [ ]:
from torch.utils.data import DataLoader, RandomSampler, SequentialSampler

# Create the DataLoaders for our training and validation sets.
# We'll take training samples in random order.
train_dataloader = DataLoader(
            train_dataset,  # The training samples.
            sampler = RandomSampler(train_dataset), # Select batches randomly
            batch_size = batch_size # Trains with this batch size.
        )

: 

In [ ]:
from transformers import AdamW

# Note: AdamW is a class from the huggingface library (as opposed to pytorch)
# I believe the 'W' stands for 'Weight Decay fix"
optimizer = AdamW(model.parameters(),
                  lr = learning_rate,
                  eps = 1e-8
                )

: 

In [ ]:
from transformers import get_linear_schedule_with_warmup

# Total number of training steps is [number of batches] x [number of epochs].
# (Note that this is not the same as the number of training samples!)
total_steps = len(train_dataloader) * epochs

# Create the learning rate scheduler.
scheduler = get_linear_schedule_with_warmup(optimizer,
                                            num_warmup_steps = 0, # Default value in run_glue.py
                                            num_training_steps = total_steps)

: 

In [ ]:
import numpy as np

# Function to calculate the accuracy of our predictions vs labels
def flat_accuracy(preds, labels):
    pred_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    return np.sum(pred_flat == labels_flat) / len(labels_flat)

: 

In [ ]:
import time
import datetime

def format_time(elapsed):
    '''
    Takes a time in seconds and returns a string hh:mm:ss
    '''
    # Round to the nearest second.
    elapsed_rounded = int(round((elapsed)))

    # Format as hh:mm:ss
    return str(datetime.timedelta(seconds=elapsed_rounded))

: 

In [ ]:
import pandas as pd

# Display floats with two decimal places.
#pd.set_option('precision', 2)

# Create a DataFrame from our training statistics.
df_stats = pd.DataFrame(data=training_stats)

# Use the 'epoch' as the row index.
df_stats = df_stats.set_index('epoch')

# A hack to force the column headers to wrap (doesn't seem to work in Colab).
#df = df.style.set_table_styles([dict(selector="th",props=[('max-width', '70px')])])

# Display the table.
df_stats

: 

In [ ]:
import torch.nn.functional as F

: 

In [ ]:
# Create a DataLoader to batch our test samples for us. We'll use a sequential
# sampler this time--don't need this to be random!
prediction_sampler = SequentialSampler(test_dataset)
prediction_dataloader = DataLoader(test_dataset, sampler=prediction_sampler, batch_size=batch_size)

print('Predicting labels for {:,} test sentences...'.format(len(test_dataset)))

# Put model in evaluation mode
model.eval()

# Tracking variables
predictions , prob, true_labels = [],[], []

# Predict
for batch in prediction_dataloader:
  # Add batch to GPU
  batch = tuple(t.to(device) for t in batch)

  # Unpack the inputs from our dataloader
  b_input_ids, b_input_mask, b_labels = batch

  # Telling the model not to compute or store gradients, saving memory and
  # speeding up prediction
  with torch.no_grad():
      # Forward pass, calculate logit predictions.
      result = model(b_input_ids,
                     token_type_ids=None,
                     attention_mask=b_input_mask,
                     return_dict=True)

  logits = result.logits
  probabilities = F.softmax(logits, dim=-1)


  # Move logits and labels to CPU
  logits = logits.detach().cpu().numpy()
  probabilities = probabilities.detach().cpu().numpy()
  label_ids = b_labels.to('cpu').numpy()

  # Store predictions and true labels
  predictions.append(logits)
  prob.append(probabilities)
  true_labels.append(label_ids)

print('    DONE.')

: 

In [ ]:
# Combine the results across all batches.
flat_predictions = np.concatenate(predictions, axis=0)

# For each sample, pick the label (0 or 1) with the higher score.
flat_predictions = np.argmax(flat_predictions, axis=1).flatten()

# Combine the correct labels for each batch into a single list.
flat_true_labels = np.concatenate(true_labels, axis=0)

: 

In [ ]:
from sklearn.metrics import f1_score

# Calculate the F1
f1 = f1_score(flat_true_labels, flat_predictions)

print('F1 Score: %.3f' % f1)

: 

In [ ]:
print(classification_report(flat_true_labels, flat_predictions))

: 

In [ ]:
flat_predictions

: 

In [ ]:
prob_flat = np.concatenate(prob, axis=0)

: 

In [ ]:
test_metric, case_arr_naive, case_arr = compute_case_accuracy(testing_set_logistic,flat_predictions,prob_flat[:,1])

: 

In [ ]:
model.save_pretrained("model/binary")

: 